In [5]:
%pip install pandas
%pip install python-calamine

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import numpy as np
import pandas as pd

# ==========================================
# 1. โหลดข้อมูล (ใส่ engine='calamine' เพื่อเลี่ยง XML เสียหายจากรอบแรก)
# ==========================================
file_path = r"C:\Users\KS\Desktop\product.xlsx"
df = pd.read_excel(file_path, engine="calamine")

# ล้างช่องว่างที่อาจมองไม่เห็นในชื่อคอลัมน์ทิ้งให้หมดเพื่อความปลอดภัย
df.columns = df.columns.str.strip()

df = df[df['import'] > 0]
# ==========================================
# 2. คำนวณหา Outlier แบบประวัติตัวมันเอง (วิธีที่เสถียรที่สุด)
# ==========================================
# ใช้ .transform() เพื่อหาค่า Mean และ SD แยกกลุ่มสินค้า แต่รักษาโครงสร้างตารางเดิมไว้ร้อยเปอร์เซ็นต์
group_mean = df.groupby("product_id")["import"].transform("mean")
group_mad = df.groupby("product_id")["import"].transform("std")

# คำนวณ Z-score โดยระวังกรณีที่ค่า SD เป็น 0 (ยอดขายนิ่งสนิท)
# ถ้า SD เป็น 0 หรือหาค่าไม่ได้ ให้ Z-score เป็น 0
df["z_score"] = np.where(
    (group_mad == 0) | (group_mad.isna()),  # [IF]   ถ้า SD เป็น 0 หรือหาค่าไม่ได้
    0,  # [THEN] ให้ Z-score เป็น 0 ไปเลย (เพื่อไม่ให้สูตรคณิตศาสตร์พัง)
    (df["import"] - group_mean) / group_mad,  # [ELSE] ถ้าปกติ ก็จับลบกันแล้วหารด้วย SD ตามสูตรปกติ
)

# กรองเกณฑ์ที่เริ่มแกว่ง (ปรับตัวเลขจาก 2 เป็น 1.5 ได้ตามความไวที่ต้องการ)
df["is_outlier"] = df["z_score"].abs() > 3


# ==========================================
# 3. เจาะลึกระดับบิล (Drill-Down)
# ==========================================
# ดึงรายชื่อรหัสสินค้าทั้งหมดที่มีแถวใดแถวหนึ่งติดสถานะ Outlier
outlier_products = df[df["is_outlier"] == True]["product_id"].unique()

# ดึง "ทุกบิล" ของสินค้าในกลุ่ม outlier_products ขึ้นมาดูประวัติเปรียบเทียบ
final_report = df[df["product_id"].isin(outlier_products)].copy()

# จัดเรียงข้อมูลให้ดูง่าย: เรียงตามรหัสสินค้า และเอาบิลที่แกว่งที่สุด (Z-score สูงสุด) ขึ้นก่อน
final_report = final_report.sort_values(
    by=["product_id", "z_score"], ascending=[True, False]
)


In [26]:
excel = final_report[["DATE", "product_id", "Bill", "import", "z_score"]]

 Robust Z-score

In [3]:
%pip install scipy

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement scipy (from versions: none)

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for scipy


In [4]:
%pip install python-calamine
%pip install pandas
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [59]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import openpyxl

# ==========================================
# 1. โหลดข้อมูล (ใส่ engine='calamine' เพื่อเลี่ยง XML เสียหายจากรอบแรก)
# ==========================================
file_path = r"C:\Users\USER\Desktop\สต็อกรวม.xlsx"
df = pd.read_excel(file_path, engine="calamine")

# ล้างช่องว่างที่อาจมองไม่เห็นในชื่อคอลัมน์ทิ้งให้หมดเพื่อความปลอดภัย
df.columns = df.columns.str.strip()

# [เสริมเกราะ 1] แปลง ID ให้เป็น string ทั้งหมด ป้องกันกรณี Excel แปลงบางตัวเป็นตัวเลขแล้วกลุ่มเพี้ยน
df['product_id'] = df['product_id'].astype(str).str.strip()

# เอาเฉพาะยอดนำเข้าที่มากกว่า 0 เท่านั้น (ตัด Noise/บิลยกเลิก ออก)
df = df[df['import'] > 0]

# ==========================================
# 2. คำนวณหา Outlier 
# ==========================================
# หา IQR และคูณสเกล 1.4826 สำหรับ แผน A
q1 = df.groupby('product_id')['import'].transform(lambda x: x.quantile(0.25))
q3 = df.groupby('product_id')['import'].transform(lambda x: x.quantile(0.75))
iqr_scaled = (q3 - q1) * 1.4826

# หา Median และจำนวนบิล
group_median = df.groupby("product_id")["import"].transform("median")
group_count = df.groupby('product_id')['import'].transform('count')

# ✨ [แก้ไขจุดที่ 1] หา MAD ดิบจาก SciPy แล้วค่อยคูณสเกล 1.4826 ข้างนอก (ป้องกันการคูณเบิ้ล)
mad_raw = df.groupby("product_id")["import"].transform(stats.median_abs_deviation)
group_mad_scaled = mad_raw * 1.4826

# เงื่อนไขที่ 2 ไม่ให้พุ่งสูงเกิน 30% ของค่ากลาง
max_allowed_deviation = np.maximum(group_median * 0.3, 1.0)
group_mad_scaled = np.minimum(group_mad_scaled, max_allowed_deviation)
group_mad_scaled = np.maximum(group_mad_scaled, 1.0) # กันตัวหารเป็น 0

# เงื่อนไขการแบ่งกลุ่มสินค้า
conditions = [
    (iqr_scaled > 0) & (group_count >= 10),   # แผน A
    (iqr_scaled == 0) | (group_count < 10)    # แผน B
]

# คำนวณคะแนนดิบ Z-score
df["Robustz_score/iqr"] = np.select(conditions, [
    ((df["import"] - group_median) / iqr_scaled),       # แผน A
    ((df["import"] - group_median) / group_mad_scaled)  # แผน B (ใช้ตัวหารที่ล็อกสเกลและดักคอแล้ว)
], default=0)

# ตัดเกรด Outlier มาตรฐานเดี่ยว 99.7% เท่ากันทั้ง 2 ฝั่ง (> 3.0)
df["is_outlier"] = df['Robustz_score/iqr'].abs() > 3

# ==========================================
# 3. เจาะลึกระดับบิล (Drill-Down)
# ==========================================
# ดึงรายชื่อรหัสสินค้าทั้งหมดที่มีแถวใดแถวหนึ่งติดสถานะ Outlier
final_report = df[df["is_outlier"] == True].copy()

# ✨ [แก้ไขจุดที่ 2] จัดเรียงข้อมูลพร้อมใส่วงเล็บปิดให้สมบูรณ์
final_report = final_report.sort_values(
    by=["product_id", "Robustz_score/iqr"], ascending=[True, False]
)

In [ ]:
#final_report.to_excel(r'C:\Users\KS\Desktop\product_out1.xlsx',index=False,engine='openpyxl')

In [60]:
final_report[['DATE','Bill','product_id','import','Robustz_score/iqr']].reset_index(drop=True
                                                                                    )

,DATE,Bill,product_id,import,Robustz_score/iqr
0,14/03/2569,IBK3256903/039,กะซ้ง น้ำดื่ม 850 มล.,600.0,5.395926
1,05/04/2569,IBK3256904/017,น้ำ เพียวไลฟ์ 600มล.,300.0,8.584428
2,17/04/2569,IBK3256904/053,น้ำ เพียวไลฟ์ 600มล.,300.0,8.584428
3,04/05/2569,IBK3256905/010,น้ำ เพียวไลฟ์ 600มล.,300.0,8.584428
4,25/05/2569,IBK3256905/080,น้ำ เพียวไลฟ์ 600มล.,300.0,8.584428
5,15/06/2569,IBK3256906/041,น้ำ เพียวไลฟ์ 600มล.,300.0,8.584428
6,28/06/2569,IBK3256906/077,น้ำ เพียวไลฟ์ 600มล.,300.0,8.584428
7,05/07/2569,IBK3256907/012,น้ำ เพียวไลฟ์ 600มล.,300.0,8.584428
8,14/03/2569,IBK3256903/042,น้ำ เพียวไลฟ์ 600มล.,150.0,3.985627
9,19/03/2569,IBK3256903/054,น้ำ เพียวไลฟ์ 600มล.,150.0,3.985627


In [ ]:
#final_report[['DATE','Bill','product_id','import','Robustz_score/iqr']].reset_index(drop=True).to_excel(
   # r'C:\Users\USER\Desktop\product_out1.xlsx',index=False,engine='openpyxl')
                                                                                    